In [1]:
from IPython.display import Markdown, display
import pandas as pd

from tb_macro.document import get_func_notes, build_fixed_params_table, build_prior_ranges_table
from tb_macro.acf import add_acf
from tb_macro.health_system import (
    add_detection,
    add_treatment_flows,
    compute_outcome_props,
    get_outcome_rates,
)
from tb_macro.inputs import (
    add_groups_to_single_pop,
    build_age_weight_lookup,
    calc_death_in_unsucc_outcomes,
    calc_tsr_from_outcomes,
    get_norm_conmat,
    get_country_indicators,
    get_country_pop,
    get_fertility_data,
    get_single_age_pop_from_ungroups,
    get_un_mortality,
    load_demography,
    load_fertility,
    load_who_outcomes,
)
from tb_macro.mixing import (
    aggregate_full_matrix_to_groups,
    build_c_matrix,
    build_s_matrix_single_age,
    get_norm_c_matrix,
    canberra_distance,
)
from tb_macro.demography import (
    add_ageing_flows,
    add_entry_births,
    add_replacement_deaths,
    inflate_oldest_death_rates,
    prepare_pop_data_for_entries,
)
from tb_macro.epi import (
    add_infection_flows,
    add_latency_flows,
    add_natural_history,
    add_seeding,
    get_base_model,
    get_latency_age_adj,
    infect_process,
    initialise_pops,
)
from tb_macro.calibration import (
    make_log_likelihood,
    get_latent_log_likelihood,
    get_notification_log_likelihood,
    get_death_log_likelihood,
    get_adult_pulm_prev,
    get_pulm_prev_log_likelihood,
    get_prev_decline_log_likelihood,
    get_infprop_log_likelihood,
    get_mixing_log_likelihood,
)
from tb_macro.outputs import (
    build_age_mapping,
    get_age_deaths,
    get_age_inc,
    get_age_notifs,
    get_age_prev,
    get_age_pulm_prev,
    get_age_rx_prev,
    map_and_regroup_output,
    regroup_full_outputs,
)
from tb_macro.parameters import BASE_PARAMS, PARAM_BOUNDS

/Users/jamestrauer/dev/tb_macroeconomics/.pixi/envs/default/lib/python3.13/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
display(Markdown("""# Demographics
The modelled population is stratified by age. 
Our demographic model is designed to exactly match total population size estimates,
be mechanistically coherent and to support a realistic population distribution by age 
to emerge from the applied demographic processes.
"""))
display(Markdown("""## Age groups"""))
display(Markdown(get_func_notes(add_ageing_flows)))

display(Markdown("""## Population data"""))
display(Markdown(
    get_func_notes(get_country_pop) + " " +
    get_func_notes(get_single_age_pop_from_ungroups) +
    """ This approach allows the population to then be
    aggregated to any requested set of age groups.
    """
    )
)
display(Markdown("""## Background mortality"""))
display(Markdown("""Age-specific death rates are formed from 
deaths and populations reported by the UN in each modelled age group. Non-TB deaths are 
applied as replacement births into the youngest uninfected group.
""" + get_func_notes(get_un_mortality) + " " + 
get_func_notes(load_demography) + "\n\n" + get_func_notes(inflate_oldest_death_rates) +
"\n\n" + get_func_notes(add_replacement_deaths)
))
display(Markdown("""## Population size"""))
display(Markdown(get_func_notes(initialise_pops)))
display(Markdown("""As described above, 
the replacement of deaths keeps background mortality from affecting 
the total population size. 
Each model run starts with an entirely uninfected population
assigned to the _Mtb_-naive state.
""" + get_func_notes(prepare_pop_data_for_entries)
))
display(Markdown(get_func_notes(add_entry_births)))

display(Markdown("""# Natural history of tuberculosis"""))
display(Markdown(get_func_notes(get_base_model)))
display(Markdown("""## Seeding"""))
display(Markdown(get_func_notes(add_seeding)))
display(Markdown("""## Infection"""))
display(Markdown(get_func_notes(add_infection_flows) + " " + get_func_notes(infect_process)))
display(Markdown("""## Progression through infection and disease states"""))
display(Markdown(get_func_notes(get_latency_age_adj) + " " + get_func_notes(add_latency_flows) + " " + get_func_notes(add_natural_history)))

display(Markdown("# Population mixing"))
display(Markdown("## Mixing kernel"))
display(Markdown(get_func_notes(build_s_matrix_single_age) + " " + get_func_notes(get_fertility_data) + " " + get_func_notes(load_fertility)))
display(Markdown("## Aggregation"))
display(Markdown(get_func_notes(build_age_weight_lookup) + " " + get_func_notes(aggregate_full_matrix_to_groups) + " " + get_func_notes(build_c_matrix)))
display(Markdown("## Normalisation"))
display(Markdown(get_func_notes(get_norm_c_matrix)))

display(Markdown("""# Health system"""))
display(Markdown("## Detection"))
display(Markdown(get_func_notes(add_detection)))
display(Markdown("## Treatment"))
display(Markdown(get_func_notes(calc_tsr_from_outcomes) + " " + get_func_notes(calc_death_in_unsucc_outcomes) + " " + get_func_notes(compute_outcome_props)))
display(Markdown(get_func_notes(add_treatment_flows)))
display(Markdown(get_func_notes(get_outcome_rates)))

display(Markdown("# Active case finding"))
display(Markdown(get_func_notes(add_acf)))

display(Markdown("# Model outputs"))
output_defs = {
    "Incidence": get_func_notes(get_age_inc),
    "Prevalence": get_func_notes(get_age_prev),
    "Pulmonary prevalence": get_func_notes(get_age_pulm_prev),
    "Treatment prevalence": get_func_notes(get_age_rx_prev),
    "Notifications": get_func_notes(get_age_notifs),
    "TB deaths": get_func_notes(get_age_deaths),
}
output_defs = {"Definition": {k: " ".join(v.split()) for k, v in output_defs.items()}}
display(Markdown(pd.DataFrame(output_defs).rename_axis("Indicator").to_markdown() + "\n\n: {tbl-colwidths=\"[30,70]\"}"))
display(Markdown("## Mapping to output age groups"))
display(Markdown(get_func_notes(build_age_mapping) + " " + get_func_notes(map_and_regroup_output) + " " + get_func_notes(regroup_full_outputs)))

display(Markdown("# Calibration\n## Likelihood construction"))
display(Markdown(get_func_notes(make_log_likelihood)))
display(Markdown("## Target calculations"))
display(Markdown("### Prevalence of infection"))
display(Markdown(get_func_notes(get_latent_log_likelihood)))
display(Markdown("### Notifications"))
display(Markdown(get_func_notes(get_notification_log_likelihood)))
display(Markdown("### Deaths"))
display(Markdown(
    get_func_notes(load_who_outcomes) + " " +
    get_func_notes(get_country_indicators) + " " +
    get_func_notes(get_death_log_likelihood)
))
display(Markdown("### Pulmonary prevalence"))
display(Markdown(
    get_func_notes(get_adult_pulm_prev) + " " +
    get_func_notes(get_pulm_prev_log_likelihood) +
    "\n\n" + get_func_notes(get_prev_decline_log_likelihood) +
    "\n\n" + get_func_notes(get_infprop_log_likelihood)
))
display(Markdown("### Mixing matrix"))
display(Markdown(
    get_func_notes(get_mixing_log_likelihood) +
    "\n\n" + get_func_notes(canberra_distance)
))

# Demographics
The modelled population is stratified by age. 
Our demographic model is designed to exactly match total population size estimates,
be mechanistically coherent and to support a realistic population distribution by age 
to emerge from the applied demographic processes.


## Age groups

The population is stratified into age groups with lower bounds of: 
0, 3, 5, 10, 15, 18, 40, 65 years. People move from each group to the next at a
constant rate equal to the reciprocal of the with of the group 
they are exiting in years, such that the mean time spent in 
each age group matches its width. The oldest group has 
no ageing outflow; exit from this group occurs only through death.

## Population data

Population counts are taken from the UN World Population
Prospects, by calendar year and age group, for country: VNM. UN age-group counts are recorded in thousands and are 
distributed uniformly across the single years of age 
contained by each group before further processing 
to modelled age brackets (with open-ended groups extended 
to 120 years). This approach allows the population to then be
    aggregated to any requested set of age groups.
    

...

WHO estimates of TB deaths with and without HIV are summed
to create a single mortality series. Burden estimates are offset by 0.5 of a year 
to sit at mid-year. Deaths occurring in the community and during treatment are 
summed for comparison.

As for notifications, these counts are
compared on the log scale with a standard deviation of 0.1, 
and averaged over the years for which estimates are available.

### Pulmonary prevalence

Prevalence is calculated to approximate the quantity ascertained by a
bacteriologically-confirmed prevalence survey, and so is restricted to
adults, taken here as those aged 15 years and over.
Three groups contribute to the numerator: all persons with active disease
in the high infectiousness stratum, a proportion of persons in the low
infectiousness stratum given by the "proportion of low-infectious TB that is bacteriologically positive", and
everyone currently receiving treatment. Clinical status does not
enter this calculation, with subclinical and clinical disease contributing
on the same basis. The denominator is the total 
adult population at the same time point. Modelled adult prevalence of bacteriologically-confirmed pulmonary TB was
compared against the most recent (second) Vietnamese national prevalence survey
(PLOS One). This study reports prevalent cases per 100,000 population 
as a proportion of the adult population. Being a proportion, 
this quantity was compared on the log-odds scale with
a standard deviation of 0.2.

The two survey rounds reported and compared in Nguyen et al. 
(Emerging Infectious Diseases) are used to target the decline in 
prevalence rather than absolute values.
The ratio of the later to the earlier adult pulmonary prevalence estimate
is taken, such that this rate of decline target constrains the trend in 
a single quantity over time, without reference to the absolute prevalence value.
The ratio is compared on the log scale with a
 standard deviation of 0.1.

The proportion of prevalent adult TB that is highly infectious was
compared against the equivalent proportion from the second Vietnamese
national prevalence survey.
The numerator is the high infectiousness stratum of the active
compartment and the denominator is total adult prevalence, as defined
above. This target therefore constrains how prevalent disease is
distributed across the infectiousness strata, without
constraining the overall size of the prevalent pool.
Being a proportion, this quantity is compared on the log-odds scale with
a standard deviation of 0.2.

### Mixing matrix

The modelled age-mixing matrix in 2025 was compared
against a synthetic contact matrix for VNM, generated by projecting
POLYMOD contact patterns onto the country's age structure using the
`conmat` R package. Both matrices are normalised by their spectral
radius before comparison, so that only the relative pattern of
age-to-age mixing is targeted.

The discrepancy is summarised by the Canberra distance, which gives
comparable weight to low- and high-contact cells. This distance is
treated as a residual around zero,
$d \sim \mathcal{N}(0, \sigma_{m})$, where $\sigma_{m}$ is the
"mixing matrix distance standard deviation". Unlike the fixed error scales used for the
epidemiological targets, $\sigma_{m}$ is estimated during calibration
under a uniform prior. This restricted range stops the mixing penalty from becoming
arbitrarily tight or from being ignored entirely. Within that range,
$\sigma_{m}$ tracks how large a discrepancy is needed to reconcile the
synthetic contact matrix with the epidemiological data.

The Canberra distance is the sum over corresponding cells of the
absolute difference divided by the sum of the absolute values,
$$
d = \sum_{i,j}
\frac{|x_{ij} - y_{ij}|}{|x_{ij}| + |y_{ij}| + \varepsilon}
$$
with $\varepsilon =$ $1 \times 10^{-10}$ to avoid division by zero.
Because each term represents a relative discrepancy, low-contact cells
contribute comparably to high-contact cells.

## Fixed parameters

In [3]:
fixed_params = {k: v for k, v in BASE_PARAMS.items() if k not in PARAM_BOUNDS}
build_fixed_params_table(fixed_params)

,value
Parameter,
relative susceptibility of the never-infected,1
relative infectiousness of subclinical TB,0.5
relative infectiousness of low-infectious TB,0.4
progression rate under 5 years,2.4
progression rate ages 5 to 15,2
progression rate ages 15 and over,0.1
proportion of progressions that are high-infectious,0.2
containment rate under 5 years,4.4
containment rate ages 5 to 15,4.4


## Prior ranges

Pass the dict of prior bounds to show.

In [4]:
build_prior_ranges_table(PARAM_BOUNDS)

,lower,upper
Parameter,,
raw transmission rate,8,20
background mixing,0.001,0.02
assortative mixing spread,2,15
parent-child mixing strength,0.05,1
relative susceptibility of contained infection,0.2,0.8
relative susceptibility of cleared or recovered infection,0.2,0.8
relative susceptibility of never-infected children,0.2,0.8
breakdown rate,0.01,0.5
clearance rate,0.01,0.1
